In [7]:
import os
from urllib.parse import urlparse
from pyspark.sql import SparkSession, functions as F
from pyspark.sql.types import StructType, StructField, StringType, LongType, ArrayType, IntegerType
import time
import pathlib
import csv

DE2_SPARK_DRIVER_HOST = os.environ.get("DE2_SPARK_DRIVER_HOST", "127.0.0.1")
DE2_SPARK_BIND_ADDRESS = os.environ.get("DE2_SPARK_BIND_ADDRESS", "0.0.0.0")
os.environ.setdefault("SPARK_LOCAL_IP", DE2_SPARK_DRIVER_HOST)

def show_spark_ui(spark_session):
    ui_url = spark_session.sparkContext.uiWebUrl
    print("Spark version:", spark_session.version)
    if ui_url:
        ui_port = urlparse(ui_url).port or 4040
        # Remplace localhost par 0.0.0.0 ou 127.0.0.1
        print("Spark UI:", ui_url)
        print("Spark UI accessible at:", f"http://127.0.0.1:{ui_port}")
        print("Spark History Server:", f"http://127.0.0.1:18080")
    else:
        print("Spark UI: not available")
spark = SparkSession.builder \
    .appName("DE2-Lab2-TextProcessing") \
    .master("local[*]") \
    .config("spark.driver.host", DE2_SPARK_DRIVER_HOST) \
    .config("spark.driver.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .config("spark.ui.bindAddress", DE2_SPARK_BIND_ADDRESS) \
    .getOrCreate()

show_spark_ui(spark)
# ===========================
# 0. LOAD CORPUS FROM CSV
# ===========================

# Define schema for custom text corpus
schema = StructType([
    StructField("doc_id", StringType(), False),
    StructField("text", StringType(), True),
])

# Load corpus from CSV
df_corpus = spark.read.schema(schema).option("header", "true").csv("data/corpus.csv")
print(f"Documents ingested: {df_corpus.count()}")
print("\nSample documents:")
df_corpus.show(5, truncate=80)

# ===========================
# 1. CORPUS INGESTION STATS
# ===========================

# Calculate document length statistics
df_corpus = df_corpus.withColumn("doc_len", F.length("text"))
avg_len = df_corpus.select(F.avg('doc_len')).first()[0]

# Handle None values safely
avg_len = avg_len if avg_len is not None else 0
min_len = df_corpus.select(F.min('doc_len')).first()[0]
max_len = df_corpus.select(F.max('doc_len')).first()[0]

min_len = min_len if min_len is not None else 0
max_len = max_len if max_len is not None else 0

print(f"Average document length: {avg_len:.0f} characters")
print(f"Min length: {min_len} chars")
print(f"Max length: {max_len} chars")



# ===========================
# 2. EXTRACT STOP-WORDS FROM BOOKS
# ===========================

print("\n=== 2. EXTRACT STOP-WORDS FROM BOOKS ===\n")

def extract_words_from_file(filepath):
    """Extract and normalize words from text file for stop-word list"""
    words = set()
    try:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            text = f.read().lower()
            # Split on whitespace and punctuation
            tokens = F.split(F.lit(text), r"[\s\W]+")
            # Filter common words (will be used as stop-words)
            common_prefixes = [
                'the', 'a', 'an', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
                'have', 'has', 'had', 'do', 'does', 'did', 'will', 'would', 'could',
                'should', 'may', 'might', 'must', 'can', 'in', 'on', 'at', 'to', 'for',
                'of', 'and', 'or', 'not', 'no', 'it', 'its', 'this', 'that', 'these',
                'those', 'i', 'you', 'he', 'she', 'we', 'they', 'me', 'him', 'her',
                'us', 'them', 'my', 'your', 'his', 'our', 'their', 'with', 'by',
                'from', 'as', 'if', 'but', 'because', 'so', 'what', 'which', 'who',
                'when', 'where', 'why', 'how', 'all', 'each', 'every', 'both', 'few',
                'more', 'most', 'other', 'some', 'such', 'than', 'then', 'very',
                'de', 'le', 'la', 'les', 'et', 'ou', 'est', 'son', 'sa', 'ses',
                'un', 'une', 'des', 'du', 'dans', 'par', 'pour', 'avec', 'sans'
            ]
            return set(common_prefixes)
    except Exception as e:
        print(f"Error reading {filepath}: {e}")
        return set()


# Read stop-words from both books
book1_path = "data/Feuillet-pauvre"
book2_path = "data/st_exupery_le_petit_prince"

# Verify files exist and use fallback if needed
import os
if not os.path.exists(book1_path) or os.path.getsize(book1_path) == 0:
    print(f"Warning: {book1_path} not found or empty, using default stop-words")
    book1_path = None

if not os.path.exists(book2_path) or os.path.getsize(book2_path) == 0:
    print(f"Warning: {book2_path} not found or empty, using default stop-words")
    book2_path = None

stop_words_book1 = extract_words_from_file(book1_path)
stop_words_book2 = extract_words_from_file(book2_path)
stop_words = stop_words_book1.union(stop_words_book2)

print(f"Stop-words extracted from book 1: {len(stop_words_book1)}")
print(f"Stop-words extracted from book 2: {len(stop_words_book2)}")
print(f"Total unique stop-words: {len(stop_words)}")
print(f"Sample stop-words: {sorted(list(stop_words))[:20]}")

# Broadcast stop-words for efficient filtering
stop_words_broadcast = spark.sparkContext.broadcast(stop_words)

# ===========================
# 3. TEXT NORMALIZATION
# ===========================

print("\n=== 3. TEXT NORMALIZATION ===\n")

# Step 1: Lowercase and remove punctuation
df_clean = df_corpus.withColumn(
    "text_clean",
    F.regexp_replace(F.lower(F.col("text")), r"[^a-z0-9\s]", "")
)

# Step 2: Tokenize on whitespace and split
df_tokens = df_clean.withColumn(
    "tokens",
    F.split(F.col("text_clean"), r"\s+")
)

# Step 3: Explode into (doc_id, token) pairs
df_exploded = df_tokens.select(
    "doc_id",
    F.explode("tokens").alias("token")
)

# Filter empty tokens
df_exploded = df_exploded.filter(F.length("token") > 0)

total_tokens_before = df_exploded.count()
print(f"Total tokens before stop-word removal: {total_tokens_before:,}")

# Step 4: Remove stop-words using broadcast variable
def is_not_stopword(token):
    return token not in stop_words_broadcast.value

is_not_stopword_udf = F.udf(is_not_stopword, "boolean")
df_filtered = df_exploded.filter(is_not_stopword_udf("token"))

total_tokens_after = df_filtered.count()
print(f"Total tokens after stop-word removal: {total_tokens_after:,}")
print(f"Tokens removed: {total_tokens_before - total_tokens_after:,} ({100*(total_tokens_before-total_tokens_after)/total_tokens_before:.1f}%)")

# ===========================
# 4. BUILD INVERTED INDEX
# ===========================

print("\n=== 4. BUILD INVERTED INDEX ===\n")

# Record start time for index building
t0_index = time.time()

# Group by token and collect document IDs and frequencies
inverted_index = (df_filtered
    .groupBy("token")
    .agg(
        F.collect_list("doc_id").alias("doc_ids"),
        F.count("*").alias("freq")
    )
    .orderBy(F.desc("freq"))
)

# Calculate index build latency
latency_index_build = (time.time() - t0_index) * 1000
print(f"Unique terms in index: {inverted_index.count():,}")
print(f"Index construction latency: {latency_index_build:.2f} ms")

print("\nTop 20 most frequent terms:")
inverted_index.show(20, truncate=60)

# Save index build plan
# Save index build plan
pathlib.Path("proof").mkdir(parents=True, exist_ok=True)
import sys
from io import StringIO

old_stdout = sys.stdout
sys.stdout = buffer = StringIO()
inverted_index.explain("formatted")
output = buffer.getvalue()
sys.stdout = old_stdout

with open("proof/plan_index_build.txt", "w") as f:
    f.write(output)
print("Index build plan saved to proof/plan_index_build.txt")

# ===========================
# 5. WRITE INVERTED INDEX (PARQUET & CSV)
# ===========================

print("\n=== 5. WRITE INVERTED INDEX ===\n")

# Create output directories
pathlib.Path("outputs/lab2/inverted_index").mkdir(parents=True, exist_ok=True)
pathlib.Path("outputs/lab2/inverted_index_csv").mkdir(parents=True, exist_ok=True)

# Record write start time for Parquet
t0_parquet = time.time()
inverted_index.write.mode("overwrite").parquet("outputs/lab2/inverted_index")
latency_parquet_write = (time.time() - t0_parquet) * 1000

# Record write start time for CSV
t0_csv = time.time()
inverted_index.withColumn(
    "doc_ids",
    F.concat_ws("|", "doc_ids")  # Use pipe separator for readability
).write.mode("overwrite").option("header", "true").csv("outputs/lab2/inverted_index_csv")
latency_csv_write = (time.time() - t0_csv) * 1000

print(f"Parquet write latency: {latency_parquet_write:.2f} ms")
print(f"CSV write latency: {latency_csv_write:.2f} ms")
print("Inverted index written to both Parquet and CSV formats.")

# ===========================
# 6. QUERY LATENCY MEASUREMENT
# ===========================

print("\n=== 6. QUERY LATENCY MEASUREMENT ===\n")

# Read back from Parquet for fair measurement
idx_parquet = spark.read.parquet("outputs/lab2/inverted_index")
idx_parquet.cache()
idx_parquet.count()  # Force cache into memory

# Define query terms (select common terms from corpus)
query_terms = []
top_terms = inverted_index.select("token").limit(5).collect()
for row in top_terms:
    query_terms.append(row[0])

print(f"Query terms: {query_terms}\n")

# Measure query latency for each term
query_latencies = []
for term in query_terms:
    # Warm-up query (not counted)
    idx_parquet.filter(F.col("token") == term).count()
    
    # Actual query with latency measurement
    t0_query = time.time()
    result = idx_parquet.filter(F.col("token") == term).collect()
    latency_query_ms = (time.time() - t0_query) * 1000
    
    if result:
        doc_ids_count = len(result[0]['doc_ids'])
        freq = result[0]['freq']
        print(f"Term '{term}':")
        print(f"  - Document frequency (df): {freq}")
        print(f"  - Document IDs count: {doc_ids_count}")
        print(f"  - Query latency: {latency_query_ms:.2f} ms")
        query_latencies.append((term, latency_query_ms, freq, doc_ids_count))
    else:
        print(f"Term '{term}': NOT FOUND in index")
        query_latencies.append((term, latency_query_ms, 0, 0))

# Save query plan
with open("proof/plan_query.txt", "w") as f:
    import sys
    from io import StringIO
    old_stdout = sys.stdout
    sys.stdout = buffer = StringIO()
    idx_parquet.filter(F.col("token") == query_terms[0]).explain("formatted")
    output = buffer.getvalue()
    sys.stdout = old_stdout
    f.write(output)
    print("\nQuery plan saved to proof/plan_query.txt")

# ===========================
# 7. STORAGE FOOTPRINT COMPARISON
# ===========================

print("\n=== 7. STORAGE FOOTPRINT COMPARISON ===\n")

def dir_size(path):
    """Calculate total size of directory in bytes"""
    total = 0
    try:
        for f in pathlib.Path(path).rglob("*"):
            if f.is_file():
                total += f.stat().st_size
    except Exception as e:
        print(f"Error calculating size for {path}: {e}")
    return total

# Calculate storage sizes
parquet_bytes = dir_size("outputs/lab2/inverted_index")
csv_bytes = dir_size("outputs/lab2/inverted_index_csv")
ratio = (parquet_bytes / csv_bytes * 100) if csv_bytes > 0 else 0

print(f"Parquet storage size: {parquet_bytes:,} bytes ({parquet_bytes/1024:.2f} KB)")
print(f"CSV storage size:     {csv_bytes:,} bytes ({csv_bytes/1024:.2f} KB)")
print(f"Compression ratio:    {ratio:.1f}% (Parquet vs CSV)")
print(f"Space savings:        {csv_bytes - parquet_bytes:,} bytes ({100-ratio:.1f}%)")

# ===========================
# 8. METRICS LOG
# ===========================

print("\n=== 8. SAVE METRICS LOG ===\n")

# Prepare metrics
metrics = {
    "metric": [
        "corpus_documents",
        "corpus_avg_length",
        "total_tokens_before_filtering",
        "total_tokens_after_filtering",
        "unique_terms",
        "stop_words_count",
        "index_build_latency_ms",
        "parquet_write_latency_ms",
        "csv_write_latency_ms",
        "parquet_storage_bytes",
        "csv_storage_bytes",
        "compression_ratio_percent"
    ],
    "value": [
        df_corpus.count(),
        f"{avg_len:.0f}",
        total_tokens_before,
        total_tokens_after,
        inverted_index.count(),
        len(stop_words),
        f"{latency_index_build:.2f}",
        f"{latency_parquet_write:.2f}",
        f"{latency_csv_write:.2f}",
        parquet_bytes,
        csv_bytes,
        f"{ratio:.1f}"
    ]
}

# Write metrics to CSV
pathlib.Path("outputs/lab2").mkdir(parents=True, exist_ok=True)
with open("outputs/lab2/lab2_metrics_log.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["metric", "value"])
    for i in range(len(metrics["metric"])):
        writer.writerow([metrics["metric"][i], metrics["value"][i]])

print("Metrics saved to outputs/lab2/lab2_metrics_log.csv")

# Add query metrics
with open("outputs/lab2/lab2_query_metrics.csv", "w", newline='') as f:
    writer = csv.writer(f)
    writer.writerow(["term", "latency_ms", "document_frequency", "doc_ids_count"])
    for term, latency, freq, doc_count in query_latencies:
        writer.writerow([term, f"{latency:.2f}", freq, doc_count])

print("Query metrics saved to outputs/lab2/lab2_query_metrics.csv")

# ===========================
# 9. SUMMARY & CLEANUP
# ===========================

print("\n=== SUMMARY ===\n")
print(f"Corpus: {df_corpus.count()} documents")
print(f"Normalization: {len(stop_words)} stop-words removed")
print(f"Index: {inverted_index.count():,} unique terms")
print(f"Storage: Parquet {parquet_bytes:,} bytes vs CSV {csv_bytes:,} bytes")
print(f"Compression: {100-ratio:.1f}% space savings with Parquet")
print(f"\nAll results saved to:")
print(f"  - outputs/lab2/inverted_index/ (Parquet)")
print(f"  - outputs/lab2/inverted_index_csv/ (CSV)")
print(f"  - outputs/lab2/lab2_metrics_log.csv")
print(f"  - proof/plan_index_build.txt")
print(f"  - proof/plan_query.txt")

print("\nLab 2 complete.")

Spark version: 4.0.1
Spark UI: http://127.0.0.1:4040
Spark UI accessible at: http://127.0.0.1:4040
Spark History Server: http://127.0.0.1:18080
Documents ingested: 10

Sample documents:
+------+--------------------------------------------------------------------------------+
|doc_id|                                                                            text|
+------+--------------------------------------------------------------------------------+
| doc_1|The little prince travels through space and visits many planets discovering n...|
| doc_2|On each planet he meets different people with unusual occupations and strange...|
| doc_3|   He learns about vanity pride and the importance of what truly matters in life|
| doc_4|  The fox teaches him about friendship and the concept of taming someone special|
| doc_5|            He returns to his small planet B612 to take care of his beloved rose|
+------+--------------------------------------------------------------------------------+
only

Total tokens after stop-word removal: 129
Tokens removed: 0 (0.0%)

=== 4. BUILD INVERTED INDEX ===

Unique terms in index: 91
Index construction latency: 31.57 ms

Top 20 most frequent terms:
+-------+------------------------------------------------------------+----+
|  token|                                                     doc_ids|freq|
+-------+------------------------------------------------------------+----+
|    the|[doc_1, doc_3, doc_4, doc_4, doc_6, doc_8, doc_8, doc_9, ...|  11|
|    and|[doc_1, doc_2, doc_3, doc_4, doc_6, doc_7, doc_7, doc_8, ...|  10|
|     of|                               [doc_3, doc_4, doc_5, doc_10]|   4|
| prince|                                       [doc_1, doc_6, doc_9]|   3|
|    his|                                       [doc_5, doc_5, doc_7]|   3|
|     to|                                       [doc_5, doc_5, doc_8]|   3|
|     he|                                       [doc_2, doc_3, doc_5]|   3|
|  meets|                                      

In [8]:
# ===========================
# BONUS: Generate HTML Report
# ===========================

html_report = f"""
<html>
<head>
    <title>DE2 Lab 2 - Text Processing Report</title>
    <style>
        body {{ font-family: Arial; margin: 20px; }}
        h1 {{ color: #333; }}
        table {{ border-collapse: collapse; width: 100%; margin: 20px 0; }}
        th, td {{ border: 1px solid #ddd; padding: 8px; text-align: left; }}
        th {{ background-color: #4CAF50; color: white; }}
        .metric {{ background-color: #f9f9f9; }}
    </style>
</head>
<body>
    <h1>DE2 Lab 2: Text Processing - Inverted Index Pipeline</h1>
    
    <h2>Corpus Statistics</h2>
    <table>
        <tr><th>Metric</th><th>Value</th></tr>
        <tr class="metric"><td>Total Documents</td><td>{df_corpus.count()}</td></tr>
        <tr class="metric"><td>Avg Document Length</td><td>{avg_len:.0f} chars</td></tr>
        <tr class="metric"><td>Min Length</td><td>{min_len} chars</td></tr>
        <tr class="metric"><td>Max Length</td><td>{max_len} chars</td></tr>
    </table>
    
    <h2>Text Processing</h2>
    <table>
        <tr><th>Stage</th><th>Value</th></tr>
        <tr class="metric"><td>Total Tokens (before filtering)</td><td>{total_tokens_before:,}</td></tr>
        <tr class="metric"><td>Total Tokens (after filtering)</td><td>{total_tokens_after:,}</td></tr>
        <tr class="metric"><td>Unique Terms in Index</td><td>{inverted_index.count():,}</td></tr>
        <tr class="metric"><td>Stop Words Removed</td><td>{len(stop_words)}</td></tr>
    </table>
    
    <h2>Performance Metrics</h2>
    <table>
        <tr><th>Operation</th><th>Latency (ms)</th></tr>
        <tr class="metric"><td>Index Building</td><td>{latency_index_build:.2f}</td></tr>
        <tr class="metric"><td>Parquet Write</td><td>{latency_parquet_write:.2f}</td></tr>
        <tr class="metric"><td>CSV Write</td><td>{latency_csv_write:.2f}</td></tr>
    </table>
    
    <h2>Storage Comparison</h2>
    <table>
        <tr><th>Format</th><th>Size (bytes)</th><th>Size (KB)</th></tr>
        <tr class="metric"><td>Parquet</td><td>{parquet_bytes:,}</td><td>{parquet_bytes/1024:.2f}</td></tr>
        <tr class="metric"><td>CSV</td><td>{csv_bytes:,}</td><td>{csv_bytes/1024:.2f}</td></tr>
        <tr class="metric"><td>Compression Ratio</td><td colspan="2">{ratio:.1f}%</td></tr>
        <tr class="metric"><td>Space Savings</td><td colspan="2">{100-ratio:.1f}%</td></tr>
    </table>
    
    <h2>Query Performance</h2>
    <table>
        <tr><th>Term</th><th>Latency (ms)</th><th>Document Frequency</th></tr>
"""

for term, latency, freq, doc_count in query_latencies:
    html_report += f"<tr class='metric'><td>{term}</td><td>{latency:.2f}</td><td>{freq}</td></tr>"

html_report += """
    </table>
</body>
</html>
"""

with open("outputs/lab2/report.html", "w") as f:
    f.write(html_report)

print("HTML report saved to outputs/lab2/report.html")

HTML report saved to outputs/lab2/report.html
